In [35]:
!pip install -q dgl

In [36]:
# CELL 2 — FIXED YELPCHI SETUP
# We bypass DGL because Kaggle Python 3.12 causes DGL dependency issues.
# This downloads the exact official FraudYelp dataset used by DGL.

import os
import zipfile
import urllib.request
import scipy.io as sio
import scipy.sparse as sp
import numpy as np
import pandas as pd

URL = "https://data.dgl.ai/dataset/FraudYelp.zip"

zip_path = "/kaggle/working/FraudYelp.zip"
extract_dir = "/kaggle/working/FraudYelp"

os.makedirs(extract_dir, exist_ok=True)

# Download official dataset
if not os.path.exists(zip_path):
    print("Downloading official YelpChi dataset...")
    urllib.request.urlretrieve(URL, zip_path)

# Extract
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# Find YelpChi.mat automatically
mat_path = None

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file == "YelpChi.mat":
            mat_path = os.path.join(root, file)

if mat_path is None:
    raise FileNotFoundError("YelpChi.mat was not found.")

print("Dataset downloaded successfully.")
print("YelpChi path:", mat_path)

# Load actual dataset
yelp = sio.loadmat(mat_path)

print("\nDataset keys:")
print([k for k in yelp.keys() if not k.startswith("__")])

Dataset downloaded successfully.
YelpChi path: /kaggle/working/FraudYelp/YelpChi.mat

Dataset keys:
['homo', 'net_rur', 'net_rtr', 'net_rsr', 'features', 'label']


In [37]:
# CELL 3 — VERIFY ACTUAL YELPCHI DATA

labels = np.asarray(yelp["label"]).reshape(-1)
features = yelp["features"]

net_rur = yelp["net_rur"]
net_rtr = yelp["net_rtr"]
net_rsr = yelp["net_rsr"]

print("===== YELPCHI RAW DATA CHECK =====")

print("Labels shape:", labels.shape)
print("Features shape:", features.shape)

print("\nRelation matrix shapes:")
print("R-U-R:", net_rur.shape)
print("R-T-R:", net_rtr.shape)
print("R-S-R:", net_rsr.shape)

print("\nStored relation entries:")
print("R-U-R nnz:", net_rur.nnz)
print("R-T-R nnz:", net_rtr.nnz)
print("R-S-R nnz:", net_rsr.nnz)

print("\nUnique labels:")
unique, counts = np.unique(labels, return_counts=True)

for u, c in zip(unique, counts):
    print(f"Label {u}: {c}")

===== YELPCHI RAW DATA CHECK =====
Labels shape: (45954,)
Features shape: (45954, 32)

Relation matrix shapes:
R-U-R: (45954, 45954)
R-T-R: (45954, 45954)
R-S-R: (45954, 45954)

Stored relation entries:
R-U-R nnz: 98630
R-T-R nnz: 1147232
R-S-R nnz: 6805486

Unique labels:
Label 0: 39277
Label 1: 6677


In [38]:
# CELL 4 — YELPCHI STEP 2: NUMERICAL CHARACTERISTICS

import scipy.sparse as sp
import numpy as np

# ----------------------------
# Basic node/feature statistics
# ----------------------------
num_nodes = labels.shape[0]
num_features = features.shape[1]

fraud_count = int((labels == 1).sum())
normal_count = int((labels == 0).sum())

fraud_pct = (fraud_count / num_nodes) * 100
imbalance_ratio = normal_count / fraud_count

# ----------------------------
# Relation-wise stored edges
# ----------------------------
rur_stored = net_rur.nnz
rtr_stored = net_rtr.nnz
rsr_stored = net_rsr.nnz

stored_relation_total = rur_stored + rtr_stored + rsr_stored

# ----------------------------
# Build binary union of relations
# ----------------------------
union = net_rur.copy().astype(np.int8)

union = union + net_rtr.astype(np.int8)
union = union + net_rsr.astype(np.int8)

# Convert any value > 0 to 1
union.data[:] = 1
union.eliminate_zeros()

# Stored directed/bidirectional entries in union
union_stored_edges = union.nnz

# ----------------------------
# Self loops
# ----------------------------
self_loops = int(union.diagonal().astype(bool).sum())

# ----------------------------
# Unique undirected edges
# Exclude self-loops
# ----------------------------
union_no_diag = union.copy()
union_no_diag.setdiag(0)
union_no_diag.eliminate_zeros()

# count one side of symmetric adjacency only
unique_undirected_edges = sp.triu(
    union_no_diag,
    k=1
).nnz

# ----------------------------
# Print results
# ----------------------------
print("===== YELPCHI STEP 2 =====")

print("\nNODE / FEATURE STATISTICS")
print("Nodes:", num_nodes)
print("Features:", num_features)
print("Fraud nodes:", fraud_count)
print("Normal nodes:", normal_count)
print(f"Fraud percentage: {fraud_pct:.4f}%")
print(f"Imbalance ratio (normal:fraud): {imbalance_ratio:.4f}:1")

print("\nRAW RELATION ENTRIES")
print("R-U-R:", rur_stored)
print("R-T-R:", rtr_stored)
print("R-S-R:", rsr_stored)
print("Total relation-wise stored entries:", stored_relation_total)

print("\nUNION GRAPH")
print("Union stored entries:", union_stored_edges)
print("Self-loops:", self_loops)
print("Unique undirected edges excluding self-loops:", unique_undirected_edges)

===== YELPCHI STEP 2 =====

NODE / FEATURE STATISTICS
Nodes: 45954
Features: 32
Fraud nodes: 6677
Normal nodes: 39277
Fraud percentage: 14.5297%
Imbalance ratio (normal:fraud): 5.8824:1

RAW RELATION ENTRIES
R-U-R: 98630
R-T-R: 1147232
R-S-R: 6805486
Total relation-wise stored entries: 8051348

UNION GRAPH
Union stored entries: 7693958
Self-loops: 0
Unique undirected edges excluding self-loops: 3846979


In [39]:
# CELL 5 — YELPCHI STEP 3: GRAPH STRUCTURE

print("===== YELPCHI STEP 3 =====")

# All three matrices connect review nodes to review nodes
node_type = "Review"
num_node_types = 1

relations = {
    "R-U-R": "Reviews written by the same user",
    "R-T-R": "Reviews associated through same product/time relation",
    "R-S-R": "Reviews associated through same product/rating relation"
}

num_relation_types = len(relations)

graph_structure = "Homogeneous node type, multi-relational"
temporal_type = "Static"

print("Graph structure:", graph_structure)
print("Number of node types:", num_node_types)
print("Node type:", node_type)

print("\nNumber of relation types:", num_relation_types)

for relation, meaning in relations.items():
    print(f"{relation}: {meaning}")

print("\nGraph temporal type:", temporal_type)

# Verify all relation matrices operate on the same node population
print("\nVerification from actual matrices:")
print("R-U-R shape:", net_rur.shape)
print("R-T-R shape:", net_rtr.shape)
print("R-S-R shape:", net_rsr.shape)

assert net_rur.shape == (num_nodes, num_nodes)
assert net_rtr.shape == (num_nodes, num_nodes)
assert net_rsr.shape == (num_nodes, num_nodes)

print("\nAll three relations connect the same YelpChi review-node population.")

===== YELPCHI STEP 3 =====
Graph structure: Homogeneous node type, multi-relational
Number of node types: 1
Node type: Review

Number of relation types: 3
R-U-R: Reviews written by the same user
R-T-R: Reviews associated through same product/time relation
R-S-R: Reviews associated through same product/rating relation

Graph temporal type: Static

Verification from actual matrices:
R-U-R shape: (45954, 45954)
R-T-R shape: (45954, 45954)
R-S-R shape: (45954, 45954)

All three relations connect the same YelpChi review-node population.


In [41]:
# CELL 6 — YELPCHI STEP 4: ANOMALY LABEL ORIGIN

print("===== YELPCHI STEP 4 =====")

anomaly_origin = "Real-world / non-injected"
label_source = "Yelp platform filtering"
positive_label = "Filtered/spam review"
negative_label = "Recommended/legitimate review"

print("Anomaly origin:", anomaly_origin)
print("Label source:", label_source)
print("Positive class (1):", positive_label)
print("Negative class (0):", negative_label)

print(
    "\nConclusion: YelpChi does NOT use synthetically injected anomalies. "
    "The reviews are real-world Yelp reviews, while the spam/fraud label "
    "comes from Yelp's filtering system and should be treated as a proxy label."
)

===== YELPCHI STEP 4 =====
Anomaly origin: Real-world / non-injected
Label source: Yelp platform filtering
Positive class (1): Filtered/spam review
Negative class (0): Recommended/legitimate review

Conclusion: YelpChi does NOT use synthetically injected anomalies. The reviews are real-world Yelp reviews, while the spam/fraud label comes from Yelp's filtering system and should be treated as a proxy label.


In [43]:
# CELL 7 — YELPCHI STEP 5: GLOBAL HETEROPHILY

import numpy as np
import pandas as pd
import gc

print("===== YELPCHI STEP 5 =====")
print("Calculating global heterophily...\n")


def fast_global_heterophily(adj, labels, name):
    
    # COO gives direct row/column arrays
    coo = adj.tocoo(copy=False)

    # Count each undirected edge once.
    # Self-loops automatically excluded because row < col.
    mask = coo.row < coo.col

    src = coo.row[mask]
    dst = coo.col[mask]

    total_edges = len(src)

    heterophilic_edges = int(
        np.count_nonzero(labels[src] != labels[dst])
    )

    homophilic_edges = total_edges - heterophilic_edges

    global_h = (
        heterophilic_edges / total_edges
        if total_edges > 0
        else np.nan
    )

    result = {
        "Relation": name,
        "Total eligible edges": total_edges,
        "Homophilic edges": homophilic_edges,
        "Heterophilic edges": heterophilic_edges,
        "Global heterophily": global_h
    }

    # release temporary arrays
    del coo, mask, src, dst
    gc.collect()

    return result


# -------------------------
# Relation-specific results
# -------------------------

print("Processing R-U-R...")
rur_result = fast_global_heterophily(
    net_rur, labels, "R-U-R"
)

print("Processing R-T-R...")
rtr_result = fast_global_heterophily(
    net_rtr, labels, "R-T-R"
)

print("Processing R-S-R...")
rsr_result = fast_global_heterophily(
    net_rsr, labels, "R-S-R"
)


# -------------------------
# Combined union
# -------------------------

print("Processing combined union...")

# union_no_diag already exists from CELL 4
combined_result = fast_global_heterophily(
    union_no_diag,
    labels,
    "Combined union"
)


# -------------------------
# Final table
# -------------------------

heterophily_results = pd.DataFrame([
    rur_result,
    rtr_result,
    rsr_result,
    combined_result
])

print("\n===== GLOBAL HETEROPHILY RESULTS =====")

print(
    heterophily_results.to_string(
        index=False,
        formatters={
            "Global heterophily": lambda x: f"{x:.6f}"
        }
    )
)


# -------------------------
# Verification
# -------------------------

combined_edges = combined_result["Total eligible edges"]

print("\nVERIFICATION")
print("Combined unique undirected edges:", combined_edges)
print("Expected from Step 2:", unique_undirected_edges)

assert combined_edges == unique_undirected_edges

print("PASS: Combined edge count matches Step 2.")


# -------------------------
# Save
# -------------------------

heterophily_results.to_csv(
    "/kaggle/working/yelpchi_global_heterophily.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/yelpchi_global_heterophily.csv")

===== YELPCHI STEP 5 =====
Calculating global heterophily...

Processing R-U-R...
Processing R-T-R...
Processing R-S-R...
Processing combined union...

===== GLOBAL HETEROPHILY RESULTS =====
      Relation  Total eligible edges  Homophilic edges  Heterophilic edges Global heterophily
         R-U-R                 49315             49139                 176           0.003569
         R-T-R                573616            435564              138052           0.240670
         R-S-R               3402743           2627626              775117           0.227792
Combined union               3846979           2973887              873092           0.226955

VERIFICATION
Combined unique undirected edges: 3846979
Expected from Step 2: 3846979
PASS: Combined edge count matches Step 2.

Saved:
/kaggle/working/yelpchi_global_heterophily.csv


In [44]:
# CELL 8 — YELPCHI STEP 6: LOCAL HETEROPHILY

import numpy as np
import pandas as pd

print("===== YELPCHI STEP 6 =====")
print("Calculating local heterophily...\n")

labels_int = labels.astype(np.int32)
fraud_indicator = (labels_int == 1).astype(np.int32)


def local_heterophily_summary(adj, labels, name):
    """
    Local heterophily for node i:
    different-label neighbours / total neighbours

    Nodes with degree 0 are excluded from mean/median/std
    because local heterophily is undefined for them.
    """

    A = adj.tocsr(copy=False)

    # Degree of every node
    degree = np.diff(A.indptr).astype(np.int64)

    # Number of fraud-labelled neighbours of every node
    fraud_neighbors = np.asarray(
        A.dot(fraud_indicator)
    ).reshape(-1).astype(np.int64)

    # Different-label neighbour count
    different_neighbors = np.where(
        labels == 0,
        fraud_neighbors,              # benign node -> fraud neighbours differ
        degree - fraud_neighbors      # fraud node -> benign neighbours differ
    )

    valid = degree > 0

    local_h = np.full(len(labels), np.nan, dtype=np.float64)

    local_h[valid] = (
        different_neighbors[valid] /
        degree[valid]
    )

    values = local_h[valid]

    fraud_valid = valid & (labels == 1)
    benign_valid = valid & (labels == 0)

    result = {
        "Relation": name,
        "Nodes total": len(labels),
        "Nodes with degree > 0": int(valid.sum()),
        "Isolated nodes": int((degree == 0).sum()),
        "Mean local H": float(np.mean(values)),
        "Median local H": float(np.median(values)),
        "Std local H": float(np.std(values, ddof=0)),
        "Min local H": float(np.min(values)),
        "Q1 local H": float(np.percentile(values, 25)),
        "Q3 local H": float(np.percentile(values, 75)),
        "Max local H": float(np.max(values)),
        "Fraud-node mean H": float(np.mean(local_h[fraud_valid])),
        "Benign-node mean H": float(np.mean(local_h[benign_valid]))
    }

    return result, local_h, degree, different_neighbors


# ---------------------------
# Relation-specific
# ---------------------------

print("Processing R-U-R...")
rur_local, _, _, _ = local_heterophily_summary(
    net_rur, labels, "R-U-R"
)

print("Processing R-T-R...")
rtr_local, _, _, _ = local_heterophily_summary(
    net_rtr, labels, "R-T-R"
)

print("Processing R-S-R...")
rsr_local, _, _, _ = local_heterophily_summary(
    net_rsr, labels, "R-S-R"
)

print("Processing combined union...")
combined_local, local_h, degree, different_neighbors = \
    local_heterophily_summary(
        union_no_diag,
        labels,
        "Combined union"
    )


# ---------------------------
# Summary table
# ---------------------------

local_summary = pd.DataFrame([
    rur_local,
    rtr_local,
    rsr_local,
    combined_local
])

print("\n===== LOCAL HETEROPHILY SUMMARY =====")

display_cols = [
    "Relation",
    "Nodes with degree > 0",
    "Isolated nodes",
    "Mean local H",
    "Median local H",
    "Std local H",
    "Fraud-node mean H",
    "Benign-node mean H"
]

print(
    local_summary[display_cols].to_string(
        index=False,
        formatters={
            "Mean local H": lambda x: f"{x:.6f}",
            "Median local H": lambda x: f"{x:.6f}",
            "Std local H": lambda x: f"{x:.6f}",
            "Fraud-node mean H": lambda x: f"{x:.6f}",
            "Benign-node mean H": lambda x: f"{x:.6f}"
        }
    )
)


# ---------------------------
# Extra combined statistics
# ---------------------------

print("\n===== COMBINED UNION — FULL STATS =====")

for key in [
    "Mean local H",
    "Median local H",
    "Std local H",
    "Min local H",
    "Q1 local H",
    "Q3 local H",
    "Max local H",
    "Fraud-node mean H",
    "Benign-node mean H"
]:
    print(f"{key}: {combined_local[key]:.6f}")


# ---------------------------
# Save summary
# ---------------------------

local_summary.to_csv(
    "/kaggle/working/yelpchi_local_heterophily_summary.csv",
    index=False
)


# Save node-level combined values
node_local_results = pd.DataFrame({
    "node_id": np.arange(len(labels)),
    "label": labels,
    "degree": degree,
    "different_label_neighbors": different_neighbors,
    "local_heterophily": local_h
})

node_local_results.to_csv(
    "/kaggle/working/yelpchi_local_heterophily_nodes.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/yelpchi_local_heterophily_summary.csv")
print("/kaggle/working/yelpchi_local_heterophily_nodes.csv")

===== YELPCHI STEP 6 =====
Calculating local heterophily...

Processing R-U-R...
Processing R-T-R...
Processing R-S-R...
Processing combined union...

===== LOCAL HETEROPHILY SUMMARY =====
      Relation  Nodes with degree > 0  Isolated nodes Mean local H Median local H Std local H Fraud-node mean H Benign-node mean H
         R-U-R                  23831           22123     0.007658       0.000000    0.081438          0.074979           0.004035
         R-T-R                  45432             522     0.238627       0.142857    0.267105          0.820110           0.139727
         R-S-R                  45914              40     0.231035       0.126050    0.253776          0.795182           0.135151
Combined union                  45941              13     0.230221       0.128866    0.251982          0.805288           0.132480

===== COMBINED UNION — FULL STATS =====
Mean local H: 0.230221
Median local H: 0.128866
Std local H: 0.251982
Min local H: 0.000000
Q1 local H: 0.096154
Q3

In [45]:
# CELL 9 — YELPCHI STEP 7: ORIGINAL CARE-GNN SPLIT

import numpy as np
from sklearn.model_selection import train_test_split

print("===== YELPCHI STEP 7 =====")

# ============================================================
# IMPORTANT SPLIT DISTINCTION
# ============================================================
#
# Raw YelpChi.mat:
#   - does NOT contain a universal fixed train/val/test split.
#
# Original CARE-GNN repository default:
#   - Train = 40%
#   - Validation = none
#   - Test = 60%
#   - Random stratified split
#   - random_state = 2
#
# DGL FraudYelpDataset later provides a different default:
#   - 70% / 10% / 20%, seed 717
#
# For Venus's "original split" field, we record the
# original CARE-GNN repository protocol as the primary one.
# ============================================================


N = len(labels)
all_indices = np.arange(N)

# ------------------------------------------------------------
# ORIGINAL CARE-GNN SPLIT
# ------------------------------------------------------------

idx_train, idx_test = train_test_split(
    all_indices,
    test_size=0.60,
    random_state=2,
    shuffle=True,
    stratify=labels
)

# CARE-GNN's original implementation does not use
# a separate validation split in this default protocol.
idx_val = np.array([], dtype=int)


# ------------------------------------------------------------
# BASIC SPLIT INFORMATION
# ------------------------------------------------------------

print("Raw dataset fixed split: None")
print("Primary reference: Original CARE-GNN repository")
print("Split type: Random, stratified")
print("Temporal split: No")
print("Random state:", 2)

print("\nTRAIN / VALIDATION / TEST")
print("Train nodes:", len(idx_train))
print("Validation nodes:", len(idx_val))
print("Test nodes:", len(idx_test))

print("\nPERCENTAGES")
print(f"Train: {len(idx_train) / N * 100:.4f}%")
print(f"Validation: {len(idx_val) / N * 100:.4f}%")
print(f"Test: {len(idx_test) / N * 100:.4f}%")


# ------------------------------------------------------------
# VERIFY ALL NODES ARE ASSIGNED CORRECTLY
# ------------------------------------------------------------

print("\nCHECK")
total_assigned = len(idx_train) + len(idx_val) + len(idx_test)

print("Total assigned:", total_assigned)
print("Total YelpChi nodes:", N)

assert total_assigned == N
assert len(np.intersect1d(idx_train, idx_test)) == 0

print("PASS: All YelpChi nodes assigned exactly once.")


# ------------------------------------------------------------
# CLASS DISTRIBUTION
# ------------------------------------------------------------

def split_stats(name, idx):
    if len(idx) == 0:
        print(
            f"{name}: total=0, normal=0, fraud=0, "
            "fraud%=N/A"
        )
        return

    split_labels = labels[idx]

    normal = int((split_labels == 0).sum())
    fraud = int((split_labels == 1).sum())
    fraud_pct = fraud / len(idx) * 100

    print(
        f"{name}: total={len(idx)}, "
        f"normal={normal}, "
        f"fraud={fraud}, "
        f"fraud%={fraud_pct:.4f}%"
    )


print("\nCLASS DISTRIBUTION")
split_stats("Train", idx_train)
split_stats("Validation", idx_val)
split_stats("Test", idx_test)


# ------------------------------------------------------------
# STRATIFICATION CHECK
# ------------------------------------------------------------

overall_fraud_pct = (labels == 1).mean() * 100
train_fraud_pct = (labels[idx_train] == 1).mean() * 100
test_fraud_pct = (labels[idx_test] == 1).mean() * 100

print("\nSTRATIFICATION CHECK")
print(f"Overall fraud %: {overall_fraud_pct:.4f}%")
print(f"Train fraud %:   {train_fraud_pct:.4f}%")
print(f"Test fraud %:    {test_fraud_pct:.4f}%")

print(
    "PASS: Stratified splitting keeps the fraud proportion "
    "approximately constant."
)


# ------------------------------------------------------------
# SECONDARY REFERENCE — DGL
# ------------------------------------------------------------

print("\nSECONDARY SPLIT REFERENCE")
print(
    "DGL FraudYelpDataset later uses a different default: "
    "70% train / 10% validation / 20% test, random seed 717."
)

print(
    "This DGL split is NOT recorded as the original CARE-GNN split."
)


# ------------------------------------------------------------
# FINAL VALUE TO USE IN VENUS MASTER TABLE
# ------------------------------------------------------------

print("\n===== VALUE FOR MASTER DATASET TABLE =====")
print("Original/raw fixed split: None")
print("Reference implementation: CARE-GNN")
print("Train: 40%")
print("Validation: None / 0%")
print("Test: 60%")
print("Split: Random stratified")
print("Random state: 2")
print("Temporal: No")

===== YELPCHI STEP 7 =====
Raw dataset fixed split: None
Primary reference: Original CARE-GNN repository
Split type: Random, stratified
Temporal split: No
Random state: 2

TRAIN / VALIDATION / TEST
Train nodes: 18381
Validation nodes: 0
Test nodes: 27573

PERCENTAGES
Train: 39.9987%
Validation: 0.0000%
Test: 60.0013%

CHECK
Total assigned: 45954
Total YelpChi nodes: 45954
PASS: All YelpChi nodes assigned exactly once.

CLASS DISTRIBUTION
Train: total=18381, normal=15710, fraud=2671, fraud%=14.5313%
Validation: total=0, normal=0, fraud=0, fraud%=N/A
Test: total=27573, normal=23567, fraud=4006, fraud%=14.5287%

STRATIFICATION CHECK
Overall fraud %: 14.5297%
Train fraud %:   14.5313%
Test fraud %:    14.5287%
PASS: Stratified splitting keeps the fraud proportion approximately constant.

SECONDARY SPLIT REFERENCE
DGL FraudYelpDataset later uses a different default: 70% train / 10% validation / 20% test, random seed 717.
This DGL split is NOT recorded as the original CARE-GNN split.

===== 

Multi-relational homogeneous-node graph. Different repositories may use different edge-counting and split conventions. Yelp labels are platform-filtering proxy labels rather than independently verified criminal fraud labels. Raw .mat data do not provide one universally fixed benchmark split; DGL generates a random 70/10/20 split by default. Highly compatible with fraud-GNN models designed for YelpChi/Amazon-style multi-relational graphs. Raw relation storage = 8,051,348 entries; combined deduplicated undirected union = 3,846,979 unique edges.

In [46]:
# CELL 10 — YELPCHI STEP 8: LINKS, SOURCES AND LIMITATIONS

print("===== YELPCHI STEP 8 =====")

yelpchi_metadata = {
    "dataset": "YelpChi",

    "dataset_documentation":
        "https://www.dgl.ai/dgl_docs/en/0.8.x/generated/dgl.data.FraudYelpDataset.html",

    "download":
        "https://data.dgl.ai/dataset/FraudYelp.zip",

    "github":
        "https://github.com/YingtongDou/CARE-GNN",

    "anomaly_label_note":
        "Real-world, non-injected Yelp reviews. "
        "Yelp filtering is used as a proxy spam/anomaly label.",

    "original_split_note":
        "Raw YelpChi.mat contains no intrinsic fixed train/validation/test split. "
        "The original CARE-GNN repository uses a random stratified "
        "40% train / 0% validation / 60% test split with random_state=2.",

    "secondary_split_note":
        "DGL FraudYelpDataset later provides a different default "
        "70% train / 10% validation / 20% test random split with seed 717.",

    "edge_note":
        "8,051,348 raw relation-wise stored entries; "
        "7,693,958 stored entries after relation union; "
        "3,846,979 unique undirected union edges after reverse-edge "
        "deduplication and self-loop exclusion.",

    "heterophily_protocol":
        "Graph treated as undirected; reverse edge pairs counted once; "
        "self-loops excluded; all YelpChi nodes are labelled; "
        "duplicate structural edges across relations deduplicated in the "
        "combined union; relation-specific heterophily also calculated.",

    "local_h_sd_convention":
        "Population standard deviation, numpy ddof=0.",

    "compatibility":
        "Highly compatible with fraud-GNN models designed for "
        "same-node-type multi-relational YelpChi/Amazon-style graphs.",

    "limitations":
        "Yelp filtered/spam labels are proxy labels rather than independently "
        "verified criminal fraud. Different repositories use different "
        "train/test protocols and edge-counting conventions."
}

for key, value in yelpchi_metadata.items():
    print(f"{key}: {value}")

===== YELPCHI STEP 8 =====
dataset: YelpChi
dataset_documentation: https://www.dgl.ai/dgl_docs/en/0.8.x/generated/dgl.data.FraudYelpDataset.html
download: https://data.dgl.ai/dataset/FraudYelp.zip
github: https://github.com/YingtongDou/CARE-GNN
anomaly_label_note: Real-world, non-injected Yelp reviews. Yelp filtering is used as a proxy spam/anomaly label.
original_split_note: Raw YelpChi.mat contains no intrinsic fixed train/validation/test split. The original CARE-GNN repository uses a random stratified 40% train / 0% validation / 60% test split with random_state=2.
secondary_split_note: DGL FraudYelpDataset later provides a different default 70% train / 10% validation / 20% test random split with seed 717.
edge_note: 8,051,348 raw relation-wise stored entries; 7,693,958 stored entries after relation union; 3,846,979 unique undirected union edges after reverse-edge deduplication and self-loop exclusion.
heterophily_protocol: Graph treated as undirected; reverse edge pairs counted once

In [47]:
# CELL 11 — YELPCHI STEP 9: FINAL COMMON TABLE ROW

import pandas as pd

yelpchi_final = pd.DataFrame([{
    "Dataset": "YelpChi",
    "Domain": "Online review spam / fraud detection",

    # Scale
    "Nodes": 45954,
    "Edges": 3846979,
    "Features": 32,

    # Labels / imbalance
    "Fraud / anomaly nodes": 6677,
    "Normal nodes": 39277,
    "Fraud %": 14.5297,
    "Imbalance ratio (Normal:Fraud)": "5.8824:1",

    # Graph structure
    "Graph type": "Homogeneous node type, multi-relational",
    "Node types": 1,
    "Node type": "Review",
    "Relation types": 3,

    "Relations":
        "R-U-R: same user; "
        "R-T-R: same business/product and same month; "
        "R-S-R: same business/product and same star rating",

    "Static / Dynamic": "Static",

    # Fraud/anomaly origin
    "Anomaly origin":
        "Real-world / non-injected; Yelp filtering used as proxy spam label",

    # Global heterophily
    "Global heterophily": 0.226955,

    # Local heterophily
    "Local heterophily mean": 0.230221,
    "Local heterophily median": 0.128866,
    "Local heterophily std": 0.251982,

    "Fraud-node mean local H": 0.805288,
    "Benign-node mean local H": 0.132480,
    "Isolated nodes": 13,

    # Original/reference split
    "Raw fixed split": "None",
    "Reference implementation": "CARE-GNN",
    "Train split": "40%",
    "Validation split": "None / 0%",
    "Test split": "60%",
    "Split type": "Random, stratified",
    "Split seed / random_state": 2,
    "Temporal split": "No",

    # Sources
    "Dataset / documentation link":
        "https://www.dgl.ai/dgl_docs/en/0.8.x/generated/dgl.data.FraudYelpDataset.html",

    "Download link":
        "https://data.dgl.ai/dataset/FraudYelp.zip",

    "GitHub":
        "https://github.com/YingtongDou/CARE-GNN",

    # Notes
    "Important limitation / compatibility":
        "Raw YelpChi has no intrinsic fixed split. Original CARE-GNN uses "
        "40/0/60 stratified random splitting, while DGL later provides a "
        "different 70/10/20 default. Yelp filtering is a proxy spam label. "
        "Edge totals also differ across repositories because of relation-wise "
        "storage and bidirectional edge conventions. Highly compatible with "
        "YelpChi/Amazon-style multi-relational fraud GNNs."
}])

print("===== YELPCHI STEP 9 — FINAL ROW =====")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

display(yelpchi_final)

yelpchi_final.to_csv(
    "/kaggle/working/yelpchi_final_dataset_row.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/yelpchi_final_dataset_row.csv")

===== YELPCHI STEP 9 — FINAL ROW =====


,Dataset,Domain,Nodes,Edges,Features,Fraud / anomaly nodes,Normal nodes,Fraud %,Imbalance ratio (Normal:Fraud),Graph type,Node types,Node type,Relation types,Relations,Static / Dynamic,Anomaly origin,Global heterophily,Local heterophily mean,Local heterophily median,Local heterophily std,Fraud-node mean local H,Benign-node mean local H,Isolated nodes,Raw fixed split,Reference implementation,Train split,Validation split,Test split,Split type,Split seed / random_state,Temporal split,Dataset / documentation link,Download link,GitHub,Important limitation / compatibility
0,YelpChi,Online review spam / fraud detection,45954,3846979,32,6677,39277,14.5297,5.8824:1,"Homogeneous node type, multi-relational",1,Review,3,R-U-R: same user; R-T-R: same business/product and same month; R-S-R: same business/product and same star rating,Static,Real-world / non-injected; Yelp filtering used as proxy spam label,0.226955,0.230221,0.128866,0.251982,0.805288,0.13248,13,None,CARE-GNN,40%,None / 0%,60%,"Random, stratified",2,No,https://www.dgl.ai/dgl_docs/en/0.8.x/generated/dgl.data.FraudYelpDataset.html,https://data.dgl.ai/dataset/FraudYelp.zip,https://github.com/YingtongDou/CARE-GNN,"Raw YelpChi has no intrinsic fixed split. Original CARE-GNN uses 40/0/60 stratified random splitting, while DGL later provides a different 70/10/20 default. Yelp filtering is a proxy spam label. Edge totals also differ across repositories because of relation-wise storage and bidirectional edge conventions. Highly compatible with YelpChi/Amazon-style multi-relational fraud GNNs."



Saved:
/kaggle/working/yelpchi_final_dataset_row.csv


## YelpChi Dataset Investigation — Final

### 1. Dataset
**YelpChi**

**Domain:** Online review spam / anomaly detection.

---

### 2. Dataset characteristics

- Nodes: **45,954**
- Unique undirected union edges: **3,846,979**
- Features: **32**
- Fraud/spam nodes: **6,677**
- Normal nodes: **39,277**
- Fraud rate: **14.5297%**
- Normal:Fraud imbalance ratio: **5.8824:1**

Raw relation-wise adjacency storage contains **8,051,348 entries**.
After relation union there are **7,693,958 stored entries**, equivalent to
**3,846,979 unique undirected edges**.

---

### 3. Graph structure

- Graph type: **Homogeneous node type, multi-relational**
- Number of node types: **1**
- Node type: **Review**
- Number of relation types: **3**
- Graph setting: **Static**

Relations:

- **R-U-R:** reviews written by the same user
- **R-T-R:** reviews of the same business/product posted in the same month
- **R-S-R:** reviews of the same business/product with the same star rating

---

### 4. Fraud/anomaly source

YelpChi contains **real-world, non-injected Yelp reviews**.

The positive class represents reviews filtered by Yelp as spam/suspicious.
Therefore, the label is treated as a **platform-derived proxy spam/anomaly
label**, rather than independently verified criminal fraud.

---

### 5. Global heterophily

Protocol:

- graph treated as undirected
- reverse edge pairs counted once
- self-loops excluded
- all YelpChi nodes are labelled
- duplicate structural edges across relations deduplicated for the combined graph

Results:

- **R-U-R:** 0.003569
- **R-T-R:** 0.240670
- **R-S-R:** 0.227792
- **Combined union:** **0.226955**

Combined graph:

- Total eligible edges: **3,846,979**
- Homophilic edges: **2,973,887**
- Heterophilic edges: **873,092**

---

### 6. Local heterophily

Combined union:

- Mean: **0.230221**
- Median: **0.128866**
- Standard deviation: **0.251982**
- Q1: **0.096154**
- Q3: **0.200000**
- Minimum: **0.000000**
- Maximum: **1.000000**
- Fraud-node mean: **0.805288**
- Benign-node mean: **0.132480**
- Isolated nodes: **13**

Nodes with degree zero were excluded from local-heterophily summary statistics
because their local heterophily is undefined.

Standard deviation uses the **population definition (`ddof=0`)**.

The fraud-node mean local heterophily is substantially higher than the
benign-node mean, making neighbourhood label mixing particularly relevant
for fraud-labelled reviews.

---

### 7. Original/reference split

The raw `YelpChi.mat` file does **not** contain one intrinsic fixed
train/validation/test split.

For the original/reference implementation:

**CARE-GNN**

- Train: **40%**
- Validation: **None / 0%**
- Test: **60%**
- Split type: **Random, stratified**
- Random state: **2**
- Temporal: **No**

Actual reproduced counts:

- Train: **18,381**
- Validation: **0**
- Test: **27,573**

Class distribution:

- Train fraud rate: **14.5313%**
- Test fraud rate: **14.5287%**

The later DGL `FraudYelpDataset` loader provides a different default
**70% / 10% / 20% random split with seed 717**. This is recorded as a
secondary implementation convention and is not treated as the original
CARE-GNN split.

---

### 8. Sources

Dataset documentation:  
https://www.dgl.ai/dgl_docs/en/0.8.x/generated/dgl.data.FraudYelpDataset.html

Dataset download:  
https://data.dgl.ai/dataset/FraudYelp.zip

CARE-GNN repository:  
https://github.com/YingtongDou/CARE-GNN

---

### Important limitations / compatibility

- Yelp spam labels are platform-derived proxy labels.
- The raw dataset has no single universal fixed train/validation/test split.
- Different repositories use different split protocols.
- Reported edge counts can differ depending on bidirectional storage,
  relation union and duplicate-edge treatment.
- YelpChi is highly compatible with fraud-GNN methods developed for
  same-node-type multi-relational graphs.

No model-performance comparison was performed in this dataset-investigation task.

In [48]:
# CELL 12 — YELPCHI STEP 10: FINAL CORRECTED EXPORT PACKAGE

import os
import json
import shutil
import pandas as pd

print("===== YELPCHI STEP 10 =====")

output_dir = "/kaggle/working/yelpchi_dataset_investigation"
os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 1. FINAL DATASET TABLE ROW
# ============================================================

yelpchi_final.to_csv(
    os.path.join(output_dir, "yelpchi_final_dataset_row.csv"),
    index=False
)


# ============================================================
# 2. GLOBAL HETEROPHILY RESULTS
# ============================================================

heterophily_results.to_csv(
    os.path.join(output_dir, "yelpchi_global_heterophily.csv"),
    index=False
)


# ============================================================
# 3. LOCAL HETEROPHILY SUMMARY
# ============================================================

local_summary.to_csv(
    os.path.join(output_dir, "yelpchi_local_heterophily_summary.csv"),
    index=False
)


# ============================================================
# 4. NODE-LEVEL LOCAL HETEROPHILY
# ============================================================

node_local_results.to_csv(
    os.path.join(output_dir, "yelpchi_local_heterophily_nodes.csv"),
    index=False
)


# ============================================================
# 5. FINAL CORRECTED METADATA
# ============================================================

metadata = {
    "dataset": "YelpChi",

    "domain":
        "Online review spam / anomaly detection",

    "nodes": 45954,

    "edges": {
        "raw_relation_entries": 8051348,
        "union_stored_entries": 7693958,
        "unique_undirected_union_edges": 3846979
    },

    "features": 32,

    "labels": {
        "fraud_anomaly_nodes": 6677,
        "normal_nodes": 39277,
        "fraud_percentage": 14.5297,
        "imbalance_ratio_normal_to_fraud": 5.8824
    },

    "graph_structure": {
        "type": "Homogeneous node type, multi-relational",
        "node_type_count": 1,
        "node_type": "Review",
        "relation_type_count": 3,

        "relations": {
            "R-U-R": "Reviews written by the same user",
            "R-T-R":
                "Reviews of the same business/product posted in the same month",
            "R-S-R":
                "Reviews of the same business/product with the same star rating"
        },

        "static_dynamic": "Static"
    },

    "anomaly_origin": {
        "classification": "Real-world / non-injected",
        "label_source": "Yelp filtering system",
        "positive_class": "Filtered/spam review",
        "negative_class": "Recommended/legitimate review",
        "caveat":
            "Platform-derived proxy spam label rather than independently "
            "verified criminal fraud."
    },

    "global_heterophily": {
        "R-U-R": 0.003569,
        "R-T-R": 0.240670,
        "R-S-R": 0.227792,
        "combined_union": 0.226955,

        "combined_total_eligible_edges": 3846979,
        "combined_homophilic_edges": 2973887,
        "combined_heterophilic_edges": 873092
    },

    "local_heterophily_combined": {
        "mean": 0.230221,
        "median": 0.128866,
        "standard_deviation": 0.251982,
        "sd_convention": "Population standard deviation; ddof=0",
        "q1": 0.096154,
        "q3": 0.200000,
        "minimum": 0.0,
        "maximum": 1.0,
        "fraud_node_mean": 0.805288,
        "benign_node_mean": 0.132480,
        "isolated_nodes": 13,
        "isolated_node_treatment":
            "Excluded from local-H summaries because degree=0 makes "
            "local heterophily undefined."
    },

    "heterophily_protocol": {
        "graph_treatment": "Undirected",
        "reverse_edges": "Counted once",
        "self_loops": "Excluded",
        "unlabelled_endpoints":
            "Not applicable because all YelpChi nodes have binary labels",
        "combined_relations":
            "Union of three relations with duplicate structural edges removed",
        "relation_specific_results": True
    },

    "original_reference_split": {
        "raw_dataset_fixed_split": False,
        "reference_implementation": "CARE-GNN",
        "train_percent": 40,
        "validation_percent": 0,
        "test_percent": 60,
        "train_nodes": 18381,
        "validation_nodes": 0,
        "test_nodes": 27573,
        "split_type": "Random stratified",
        "random_state": 2,
        "temporal": False
    },

    "secondary_split_reference": {
        "implementation": "DGL FraudYelpDataset",
        "train_percent": 70,
        "validation_percent": 10,
        "test_percent": 20,
        "split_type": "Random",
        "seed": 717,
        "note":
            "Secondary loader convention, not treated as original CARE-GNN split."
    },

    "sources": {
        "dataset_documentation":
            "https://www.dgl.ai/dgl_docs/en/0.8.x/generated/dgl.data.FraudYelpDataset.html",

        "download":
            "https://data.dgl.ai/dataset/FraudYelp.zip",

        "github":
            "https://github.com/YingtongDou/CARE-GNN"
    },

    "limitations": [
        "Yelp filtering is a proxy spam/anomaly label rather than independently verified criminal fraud.",
        "Raw YelpChi has no intrinsic fixed train/validation/test split.",
        "Different implementations use different split protocols.",
        "Reported edge totals vary depending on relation-wise storage, bidirectionality and deduplication conventions."
    ],

    "model_performance_comparison_performed": False
}


with open(
    os.path.join(output_dir, "yelpchi_metadata.json"),
    "w"
) as f:
    json.dump(metadata, f, indent=4)


# ============================================================
# 6. FINAL HUMAN-READABLE SUMMARY
# ============================================================

summary_text = """
YelpChi Dataset Investigation — Final Corrected Version

DATASET
YelpChi
Domain: Online review spam / anomaly detection

DATA CHARACTERISTICS
Nodes: 45,954
Unique undirected union edges: 3,846,979
Raw relation entries: 8,051,348
Union stored entries: 7,693,958
Features: 32
Fraud/spam nodes: 6,677
Normal nodes: 39,277
Fraud percentage: 14.5297%
Normal:Fraud imbalance ratio: 5.8824:1

GRAPH STRUCTURE
Homogeneous node type, multi-relational
Node types: 1
Node type: Review
Relation types: 3

R-U-R: same user
R-T-R: same business/product and same month
R-S-R: same business/product and same star rating

Graph setting: Static

ANOMALY SOURCE
Real-world / non-injected.
Yelp filtering provides platform-derived proxy spam/anomaly labels.

GLOBAL HETEROPHILY
R-U-R: 0.003569
R-T-R: 0.240670
R-S-R: 0.227792
Combined union: 0.226955

Combined eligible edges: 3,846,979
Combined homophilic edges: 2,973,887
Combined heterophilic edges: 873,092

LOCAL HETEROPHILY — COMBINED GRAPH
Mean: 0.230221
Median: 0.128866
Population SD (ddof=0): 0.251982
Q1: 0.096154
Q3: 0.200000
Minimum: 0.000000
Maximum: 1.000000
Fraud-node mean: 0.805288
Benign-node mean: 0.132480
Isolated nodes: 13

HETEROPHILY PROTOCOL
Graph treated as undirected.
Reverse edge pairs counted once.
Self-loops excluded.
All YelpChi nodes are labelled.
Relations were analysed separately and as a deduplicated union.
Degree-zero nodes were excluded from local-H summary statistics.

ORIGINAL / REFERENCE SPLIT
Raw YelpChi.mat fixed split: None

Original CARE-GNN reference implementation:
Train: 40%
Validation: 0%
Test: 60%
Train nodes: 18,381
Validation nodes: 0
Test nodes: 27,573
Split: random stratified
Random state: 2
Temporal: No

Secondary DGL loader convention:
70% train / 10% validation / 20% test
Random seed 717

IMPORTANT LIMITATIONS
Yelp filtering is a proxy spam label.
Raw YelpChi has no universal fixed split.
Different implementations use different split protocols.
Different edge totals may be reported depending on graph storage conventions.

No model-performance comparison was performed.
"""

with open(
    os.path.join(output_dir, "yelpchi_summary.txt"),
    "w"
) as f:
    f.write(summary_text)


# ============================================================
# 7. VALIDATION BEFORE ZIP
# ============================================================

print("\nVALIDATING EXPORTED RESULTS...")

assert len(node_local_results) == 45954

assert (
    int(node_local_results["degree"].sum())
    == 2 * 3846979
)

assert (
    int(node_local_results["different_label_neighbors"].sum())
    == 2 * 873092
)

assert abs(
    873092 / 3846979 - 0.226955229
) < 1e-8

print("PASS: Node count correct.")
print("PASS: Degree sum matches edge count.")
print("PASS: Heterophilic-neighbour sum matches global H.")
print("PASS: Global heterophily internally consistent.")


# ============================================================
# 8. CREATE NEW ZIP
# ============================================================

zip_path = shutil.make_archive(
    "/kaggle/working/YelpChi_Dataset_Investigation_FINAL",
    "zip",
    output_dir
)


# ============================================================
# 9. DISPLAY FINAL FILES
# ============================================================

print("\nFILES CREATED")

for file in sorted(os.listdir(output_dir)):
    file_path = os.path.join(output_dir, file)

    print(
        f"{file}: "
        f"{os.path.getsize(file_path):,} bytes"
    )

print("\nFINAL ZIP:")
print(zip_path)

print("\n========================================")
print("YELPCHI ALL 10 STEPS COMPLETE — FINAL")
print("========================================")

print(
    "\nNo model-performance comparison was performed. "
    "Dataset investigation only."
)

===== YELPCHI STEP 10 =====

VALIDATING EXPORTED RESULTS...
PASS: Node count correct.
PASS: Degree sum matches edge count.
PASS: Heterophilic-neighbour sum matches global H.
PASS: Global heterophily internally consistent.

FILES CREATED
yelpchi_final_dataset_row.csv: 1,559 bytes
yelpchi_global_heterophily.csv: 281 bytes
yelpchi_local_heterophily_nodes.csv: 1,497,318 bytes
yelpchi_local_heterophily_summary.csv: 783 bytes
yelpchi_metadata.json: 3,932 bytes
yelpchi_summary.txt: 2,077 bytes

FINAL ZIP:
/kaggle/working/YelpChi_Dataset_Investigation_FINAL.zip

YELPCHI ALL 10 STEPS COMPLETE — FINAL

No model-performance comparison was performed. Dataset investigation only.


In [34]:
# CHECK + FORCE CREATE FINAL ZIP

import os
import shutil
from IPython.display import FileLink, display

output_dir = "/kaggle/working/yelpchi_dataset_investigation"

final_base = "/kaggle/working/YelpChi_Dataset_Investigation_FINAL"
final_zip = final_base + ".zip"

# Remove an old FINAL zip if one somehow exists
if os.path.exists(final_zip):
    os.remove(final_zip)
    print("Removed previous FINAL zip.")

# Create it again
created_zip = shutil.make_archive(
    final_base,
    "zip",
    output_dir
)

print("\nCreated:", created_zip)
print("Exists:", os.path.exists(created_zip))
print("Size:", os.path.getsize(created_zip), "bytes")

print("\nZIP FILES CURRENTLY IN /kaggle/working:")
for f in sorted(os.listdir("/kaggle/working")):
    if f.lower().endswith(".zip"):
        print(" -", f)

# Hard verification
assert os.path.exists(final_zip), "FINAL ZIP WAS NOT CREATED"

print("\n✅ FINAL ZIP EXISTS")
print(final_zip)

# Clickable download link
display(FileLink(final_zip))

Removed previous FINAL zip.

Created: /kaggle/working/YelpChi_Dataset_Investigation_FINAL.zip
Exists: True
Size: 420606 bytes

ZIP FILES CURRENTLY IN /kaggle/working:
 - FraudYelp.zip
 - YelpChi_Dataset_Investigation.zip
 - YelpChi_Dataset_Investigation_FINAL.zip

✅ FINAL ZIP EXISTS
/kaggle/working/YelpChi_Dataset_Investigation_FINAL.zip


/kaggle/working/YelpChi_Dataset_Investigation_FINAL.zip